# Notebook 04 — KPI Calculations & Risk Analysis
**Project:** Loan Default Risk Analysis and Prediction  
**Phase:** 5 of 8  
**Objective:** Compute portfolio KPIs, build a heuristic risk score, segment borrowers into risk tiers, identify key drivers of default, and surface actionable business insights.

---

## 0. Environment Setup

In [ ]:
import sys
from pathlib import Path

PROJECT_ROOT = Path.cwd().parent
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker
import seaborn as sns

from src.data_loader import load_data
from src.data_cleaner import clean_data
from src.kpi import (
    compute_portfolio_kpis, compute_segment_kpis,
    compute_kpi_trend, print_portfolio_kpis
)
from src.risk_analysis import (
    assign_risk_tier, get_risk_tier_summary, get_key_drivers,
    plot_risk_tier_distribution, plot_risk_score_by_default,
    plot_key_drivers, plot_risk_profile_heatmap
)
from src.insights import generate_insights, print_insights
from src.config import RAW_DATA_PATH, TARGET_COLUMN, FIGURES_DIR, TABLES_DIR

sns.set_theme(style='whitegrid', palette='muted', font_scale=1.1)
plt.rcParams['figure.dpi'] = 110
FIGURES_DIR.mkdir(parents=True, exist_ok=True)
TABLES_DIR.mkdir(parents=True, exist_ok=True)

print('Environment ready.')

---
## 1. Load Clean Data

In [ ]:
df = clean_data(load_data(RAW_DATA_PATH))
print(f'Clean dataset: {df.shape[0]:,} rows x {df.shape[1]} columns')

---
## 2. Portfolio-Level KPIs

In [ ]:
print_portfolio_kpis(df)

In [ ]:
kpis = compute_portfolio_kpis(df)

# Visual KPI cards
fig, axes = plt.subplots(2, 4, figsize=(16, 5))
axes = axes.flatten()

kpi_items = [
    ('Total Loans',          f"{kpis['total_loans']:,}",         '#4a90d9'),
    ('Default Rate',         f"{kpis['default_rate_pct']}%",     '#e05c5c'),
    ('Total Defaulted',      f"{kpis['total_defaulted']:,}",     '#e05c5c'),
    ('Total Loan Value',     f"${kpis['total_loan_value']/1e9:.2f}B", '#7c5cd8'),
    ('Avg Loan Amount',      f"${kpis['avg_loan_amount']:,.0f}", '#3b82d4'),
    ('Avg Credit Score',     f"{kpis['avg_credit_score']:.0f}",  '#27ae60'),
    ('Avg Interest Rate',    f"{kpis['avg_interest_rate']}%",    '#f39c12'),
    ('Defaulted Loan Value', f"${kpis['defaulted_loan_value']/1e9:.2f}B", '#e05c5c'),
]
for ax, (label, value, color) in zip(axes, kpi_items):
    ax.set_facecolor(color)
    ax.text(0.5, 0.65, value, ha='center', va='center', fontsize=20,
            fontweight='bold', color='white', transform=ax.transAxes)
    ax.text(0.5, 0.22, label, ha='center', va='center', fontsize=10,
            color='white', alpha=0.9, transform=ax.transAxes)
    ax.set_xticks([]); ax.set_yticks([])
    for spine in ax.spines.values():
        spine.set_visible(False)

plt.suptitle('Portfolio KPI Dashboard', fontsize=14, fontweight='bold', y=1.02)
plt.tight_layout()
plt.savefig(FIGURES_DIR / 'kpi_cards.png', bbox_inches='tight')
plt.show()
print('Saved -> outputs/figures/kpi_cards.png')

---
## 3. Segment KPIs — By Employment Type

In [ ]:
emp_kpis = compute_segment_kpis(df, 'EmploymentType')
emp_kpis.to_csv(TABLES_DIR / 'kpi_by_employment.csv', index=False)
print('Saved -> outputs/tables/kpi_by_employment.csv')
emp_kpis

---
## 4. Segment KPIs — By Loan Purpose

In [ ]:
purp_kpis = compute_segment_kpis(df, 'LoanPurpose')
purp_kpis.to_csv(TABLES_DIR / 'kpi_by_loan_purpose.csv', index=False)
print('Saved -> outputs/tables/kpi_by_loan_purpose.csv')
purp_kpis

---
## 5. Segment KPIs — By Education Level

In [ ]:
edu_kpis = compute_segment_kpis(df, 'Education')
edu_kpis.to_csv(TABLES_DIR / 'kpi_by_education.csv', index=False)
print('Saved -> outputs/tables/kpi_by_education.csv')
edu_kpis

---
## 6. KPI Trend — Default Rate by Age Band

In [ ]:
age_trend = compute_kpi_trend(
    df, 'Age',
    bins=[17, 25, 35, 45, 55, 70],
    labels=['18-25', '26-35', '36-45', '46-55', '56-69']
)
age_trend.to_csv(TABLES_DIR / 'kpi_trend_age.csv', index=False)

fig, axes = plt.subplots(1, 2, figsize=(13, 4))
bars = axes[0].bar(age_trend['band'], age_trend['default_rate_pct'],
                   color='#e05c5c', edgecolor='white')
axes[0].bar_label(bars, fmt='%.1f%%', padding=3, fontsize=9)
axes[0].set_title('Default Rate by Age Band', fontweight='bold')
axes[0].set_ylabel('Default Rate (%)')
axes[0].set_ylim(0, age_trend['default_rate_pct'].max() * 1.3)

axes[1].bar(age_trend['band'], age_trend['total'], color='#4a90d9', edgecolor='white')
axes[1].set_title('Loan Count by Age Band', fontweight='bold')
axes[1].set_ylabel('Number of Loans')
axes[1].yaxis.set_major_formatter(mticker.FuncFormatter(lambda x, _: f'{int(x):,}'))

plt.suptitle('KPI Trend — Age Band Analysis', fontsize=13, fontweight='bold')
plt.tight_layout()
plt.savefig(FIGURES_DIR / 'kpi_trend_age.png', bbox_inches='tight')
plt.show()
print('Saved -> outputs/figures/kpi_trend_age.png')

---
## 7. KPI Trend — Default Rate by Income Quartile

In [ ]:
inc_q = df['Income'].quantile([0, 0.25, 0.5, 0.75, 1.0]).tolist()
inc_trend = compute_kpi_trend(
    df, 'Income', bins=inc_q,
    labels=['Q1 Low', 'Q2', 'Q3', 'Q4 High']
)
inc_trend.to_csv(TABLES_DIR / 'kpi_trend_income.csv', index=False)

fig, ax = plt.subplots(figsize=(8, 4))
bars = ax.bar(inc_trend['band'], inc_trend['default_rate_pct'],
              color='#7c5cd8', edgecolor='white')
ax.bar_label(bars, fmt='%.1f%%', padding=3)
ax.set_title('Default Rate by Income Quartile', fontweight='bold')
ax.set_xlabel('Income Quartile')
ax.set_ylabel('Default Rate (%)')
ax.set_ylim(0, inc_trend['default_rate_pct'].max() * 1.3)
plt.tight_layout()
plt.savefig(FIGURES_DIR / 'kpi_trend_income.png', bbox_inches='tight')
plt.show()
print('Saved -> outputs/figures/kpi_trend_income.png')

---
## 8. Risk Scoring & Tier Assignment

In [ ]:
df_tiers = assign_risk_tier(df)
print('Risk score range:', df_tiers['RiskScore'].min(), '—', df_tiers['RiskScore'].max())
print('Risk tier value counts:')
print(df_tiers['RiskTier'].value_counts().to_string())

---
## 9. Risk Tier Summary Table

In [ ]:
tier_summary = get_risk_tier_summary(df_tiers)
tier_summary.to_csv(TABLES_DIR / 'risk_tier_summary.csv', index=False)
print('Saved -> outputs/tables/risk_tier_summary.csv')
tier_summary

---
## 10. Risk Tier Distribution Chart

In [ ]:
fig = plot_risk_tier_distribution(df_tiers)
fig.savefig(FIGURES_DIR / 'risk_tier_distribution.png', bbox_inches='tight')
plt.show()
print('Saved -> outputs/figures/risk_tier_distribution.png')

---
## 11. Risk Score vs Actual Default

In [ ]:
fig = plot_risk_score_by_default(df_tiers)
fig.savefig(FIGURES_DIR / 'risk_score_by_default.png', bbox_inches='tight')
plt.show()
print('Saved -> outputs/figures/risk_score_by_default.png')

---
## 12. Key Drivers of Default

In [ ]:
drivers = get_key_drivers(df)
drivers.to_csv(TABLES_DIR / 'key_drivers.csv', index=False)
print('Saved -> outputs/tables/key_drivers.csv')
drivers

In [ ]:
fig = plot_key_drivers(df)
fig.savefig(FIGURES_DIR / 'key_drivers.png', bbox_inches='tight')
plt.show()
print('Saved -> outputs/figures/key_drivers.png')

---
## 13. Risk Profile Heatmap — Tier x Employment

In [ ]:
fig = plot_risk_profile_heatmap(df_tiers)
fig.savefig(FIGURES_DIR / 'risk_profile_heatmap.png', bbox_inches='tight')
plt.show()
print('Saved -> outputs/figures/risk_profile_heatmap.png')

---
## 14. Business Insights & Recommendations

In [ ]:
print_insights(df)

In [ ]:
insights = generate_insights(df)
insights_df = pd.DataFrame([
    {'Priority': i['priority'], 'Category': i['category'],
     'Finding': i['finding'], 'Recommendation': i['recommendation']}
    for i in insights
])
insights_df.to_csv(TABLES_DIR / 'business_insights.csv', index=False)
print('Saved -> outputs/tables/business_insights.csv')
insights_df

---
## 15. Phase 5 Summary

| KPI | Value |
|---|---|
| Total loans | *(fill after run)* |
| Default rate | *(fill after run)* |
| Total loan value | *(fill after run)* |
| Defaulted loan value | *(fill after run)* |
| Top numeric driver | *(fill after run)* |
| Highest-risk employment | *(fill after run)* |
| Very High tier default rate | *(fill after run)* |

---
**Next:** Notebook 05 — Feature Engineering & ML Model Training